# Pirâmide

**Módulo 06 — Geometria Espacial** · Notebook 10 de 18

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/06_geometria_espacial/0610a_piramide.ipynb)

**Pré-requisitos**
- Módulo 06 — Geometria Espacial: [`0603a_perpendicularidade.ipynb`](0603a_perpendicularidade.ipynb) (especialmente as três perpendiculares), [`0608a_prismas_e_paralelepipedos.ipynb`](0608a_prismas_e_paralelepipedos.ipynb) e [`0609a_volume_e_principio_de_cavalieri.ipynb`](0609a_volume_e_principio_de_cavalieri.ipynb)
- Módulo 04 — Geometria Plana: [`0416a_triangulos_retangulos.ipynb`](../04_geometria_plana/0416a_triangulos_retangulos.ipynb) (Pitágoras), [`0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb`](../04_geometria_plana/0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb) (polígonos regulares e apótema) e [`0420a_areas_de_superficies_planas.ipynb`](../04_geometria_plana/0420a_areas_de_superficies_planas.ipynb) (áreas)
- Módulo 05 — Trigonometria: [`0509a_triangulos_retangulos.ipynb`](../05_trigonometria/0509a_triangulos_retangulos.ipynb) (razões trigonométricas)

**Objetivos de aprendizagem** — ao final deste notebook você será capaz de:
- identificar os elementos de uma pirâmide e reconhecer uma pirâmide regular;
- usar as relações métricas da pirâmide regular (altura, apótemas, aresta lateral);
- calcular as áreas lateral e total de uma pirâmide;
- explicar e usar a fórmula do volume da pirâmide, $\frac{1}{3}$ da área da base vezes a altura, e calcular a altura, a área e o volume do tetraedro regular.

**Tempo estimado:** 80 a 100 minutos

### Antes de começar

Rode a célula abaixo antes de qualquer outra. Ela carrega o `utils_3d.py`, o arquivo com as funções de desenho deste módulo.

In [1]:
# Célula de preparação: deixa o notebook pronto para rodar no seu computador ou no Colab.
import sys

if "google.colab" in sys.modules:
    # No Colab, o notebook chega sozinho; baixamos o arquivo de desenhos do repositório.
    import urllib.request
    url = "https://raw.githubusercontent.com/leandrofreiredealmeida/curso_matematica_python/main/06_geometria_espacial/utils_3d.py"
    urllib.request.urlretrieve(url, "utils_3d.py")

import utils_3d as u3d
from ipywidgets import interact
import ipywidgets as widgets
import math
import numpy as np

## Quantas pirâmides de água enchem uma caixa?

Imagine dois recipientes com a **mesma base** quadrada e a **mesma altura**: uma caixa (um prisma) e um "copo" em forma de pirâmide. Você enche o copo-pirâmide de água e despeja na caixa. Repete. Repete de novo.

Quantas vezes é preciso despejar para encher a caixa? 

🕹️ Faça uma aposta antes de usar o widget: duas vezes? Duas e meia? Três? Quatro?

In [ ]:
def despejar_agua(despejos: int, mostrar_piramide_cheia: bool) -> None:
    """Despeja pirâmides cheias de água na caixa e mostra o nível da água."""
    copo = u3d.piramide_regular(4, 2, 3)
    # A caixa fica 5 unidades ao lado do copo, só para desenhar ao lado.
    base_da_caixa = u3d.poligono_regular(4, lado=2, centro=(5, 0, 0))
    caixa = u3d.prisma(base_da_caixa, (0, 0, 3))

    volume_caixa = 2 * 2 * 3
    volume_piramide = u3d.medir_volume(*u3d.piramide_regular(4, 2, 3))  # por enquanto, só medimos
    nivel = despejos * volume_piramide / (2 * 2)

    fig = u3d.criar_figura(titulo="O copo-pirâmide e a caixa")
    fig = u3d.desenhar_poliedro(fig, *copo, opacidade=0.15)
    fig = u3d.desenhar_poliedro(fig, *caixa, opacidade=0.15)
    if mostrar_piramide_cheia:
        fig = u3d.desenhar_poliedro(
            fig, *copo, cor=u3d.CORES["plano"], opacidade=0.5, mostrar_arestas=False
        )
    # Sem nenhum despejo a caixa está vazia, e não há água para desenhar.
    if despejos > 0:
        agua = u3d.prisma(base_da_caixa, (0, 0, nivel))
        fig = u3d.desenhar_poliedro(
            fig, *agua, cor=u3d.CORES["plano"], opacidade=0.5, mostrar_arestas=False
        )
    fig = u3d.adicionar_rotulo(fig, (0, 0, 3.6), "copo")
    fig = u3d.adicionar_rotulo(fig, (5, 0, 3.6), "caixa")
    u3d.mostrar(fig)

    print(f"Nível da água na caixa: {nivel:g} (a altura da caixa é 3)")
    print(
        f"Volume despejado: {despejos} × {volume_piramide:g} = "
        f"{despejos * volume_piramide:g} (a caixa comporta {volume_caixa})"
    )
    if despejos == 3:
        print("Três pirâmides cheias enchem exatamente a caixa!")


interact(
    despejar_agua,
    despejos=widgets.IntSlider(
        min=0, max=3, step=1, value=0, description="despejos",
        continuous_update=False,
    ),
    mostrar_piramide_cheia=widgets.Checkbox(
        value=True, description="Mostrar a pirâmide cheia"
    ),
);

interactive(children=(IntSlider(value=0, continuous_update=False, description='despejos', max=3), Checkbox(val…

São necessárias exatamente **três** pirâmides. O volume de uma pirâmide é **um terço** do volume do prisma de mesma base e mesma altura. Esse "um terço" não é coincidência desse exemplo: vale para **toda** pirâmide. Neste notebook você vai entender por quê.

## 1. Pirâmide

### Pirâmide ilimitada

Pegue um polígono convexo num plano $\alpha$ e um ponto $V$ fora de $\alpha$. Todas as semirretas que saem de $V$ e passam pelos pontos do polígono formam uma **pirâmide ilimitada**. É a região de um **ângulo poliédrico** de vértice $V$ (notebook 06).

### Pirâmide

> 📌 **Definição.** Dado um polígono convexo num plano $\alpha$ e um ponto $V$ fora de $\alpha$, a **pirâmide** é o sólido formado por todos os segmentos que ligam $V$ aos pontos do polígono.

**Elementos**
- **Vértice** $V$ e **base** (o polígono).
- **Faces laterais:** triângulos com um vértice em $V$.
- **Arestas laterais:** os segmentos de $V$ até os vértices da base. **Arestas da base:** os lados da base.
- **Altura** $h$: a distância de $V$ ao plano da base (notebook 04).

O nome vem do polígono da base: **triangular** (também chamada **tetraedro**), **quadrangular**, **pentagonal**, **hexagonal**...

> 📎 **Conexão.** Uma pirâmide de base com $n$ lados tem $n + 1$ vértices, $2n$ arestas e $n + 1$ faces. Relação de Euler: $(n + 1) - 2n + (n + 1) = 2$. ✔

### Pirâmide regular

> 📌 **Definição.** Uma pirâmide é **regular** quando a base é um **polígono regular** e a projeção ortogonal do vértice sobre a base é o **centro** da base.

Numa pirâmide regular, as arestas laterais são todas iguais e as faces laterais são **triângulos isósceles congruentes**.

> ⚠️ **Erro comum.** Achar que basta a base ser regular. Se o vértice estiver "fora do centro", a pirâmide não é regular: as faces laterais deixam de ser congruentes.

🕹️ Mova o vértice da pirâmide abaixo e veja quando ela é regular.

In [3]:
def explorar_piramide(
    lados_da_base: int, altura: float, desvio_do_vertice: float
) -> None:
    """Desenha uma pirâmide com o vértice deslocado e diz se ela é regular."""
    base = u3d.poligono_regular(lados_da_base, lado=2)
    # O desvio é o deslocamento horizontal do vértice em relação ao centro da base.
    vertice = (desvio_do_vertice, 0, altura)
    vertices, faces = u3d.piramide(base, vertice)
    plano_da_base = ((0, 0, 0), (0, 0, 1))
    pe = u3d.pe_da_perpendicular_ao_plano(vertice, plano_da_base)

    fig = u3d.criar_figura(titulo="Vértice, base e altura")
    fig = u3d.desenhar_poliedro(fig, vertices, faces, opacidade=0.3)
    fig = u3d.desenhar_poligono(fig, base, cor=u3d.CORES["plano"], opacidade=0.5)
    fig = u3d.desenhar_segmento(
        fig, vertice, pe, rotulo="h", cor=u3d.CORES["reta"], tracejado=True
    )
    fig = u3d.desenhar_angulo_reto(
        fig, pe, vertice, (pe[0] + 1, pe[1], pe[2]), cor=u3d.CORES["reta"]
    )
    fig = u3d.desenhar_ponto(fig, (0, 0, 0), rotulo="O", cor=u3d.CORES["destaque"])
    fig = u3d.desenhar_ponto(fig, vertice, rotulo="V")
    u3d.mostrar(fig)

    # Um conjunto (set) não guarda repetições. Cada vértice aparece em várias faces e
    # cada aresta aparece em duas, mas no conjunto cada uma é contada uma vez só.
    numeros_de_vertices = set()
    arestas = set()
    for face in faces:
        for i in range(len(face)):
            numeros_de_vertices.add(face[i])
            arestas.add(frozenset((face[i], face[(i + 1) % len(face)])))
    v, a, f = len(numeros_de_vertices), len(arestas), len(faces)
    print(f"V = {v}, A = {a}, F = {f}; V - A + F = {v - a + f}")

    comprimentos = [u3d.medir_distancia(vertice, p) for p in base]
    print("Arestas laterais:", ", ".join(f"{c:.2f}" for c in comprimentos))
    if desvio_do_vertice == 0:
        print("Classificação: regular")
    else:
        print("Classificação: não regular: o vértice não está sobre o centro da base")


interact(
    explorar_piramide,
    lados_da_base=widgets.IntSlider(
        min=3, max=8, step=1, value=4, description="lados da base",
        continuous_update=False,
    ),
    altura=widgets.FloatSlider(
        min=1, max=5, step=0.5, value=3, description="altura",
        continuous_update=False,
    ),
    desvio_do_vertice=widgets.FloatSlider(
        min=0, max=2, step=0.25, value=0, description="desvio do vértice",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=4, continuous_update=False, description='lados da base', max=8, min=3), …

**Pare e pense**

1. Quantos vértices, arestas e faces tem uma pirâmide hexagonal?
2. Uma pirâmide tem base quadrada, mas as arestas laterais têm comprimentos diferentes. Ela é regular?

<details>
<summary>Ver respostas</summary>

1. $7$ vértices, $12$ arestas e $7$ faces.
2. Não. Numa pirâmide regular, o vértice se projeta no centro da base e todas as arestas laterais são iguais.
</details>

## 2. Relações métricas da pirâmide regular

Na pirâmide regular, quatro triângulos retângulos relacionam todas as medidas importantes. Usamos a notação:

| Medida | Símbolo | Segmento |
|---|---|---|
| aresta da base | $\ell$ | $AB$ |
| altura | $h$ | $VO$ |
| aresta lateral | $a$ | $VA$ |
| apótema da pirâmide (altura da face lateral) | $g$ | $VM$ |
| apótema da base | $m$ | $OM$ |
| raio da base (circunferência circunscrita) | $R$ | $OA$ |

**Os quatro triângulos retângulos**

| Triângulo | Ângulo reto em | Relação |
|---|---|---|
| $VOM$ | $O$ | $g^2 = h^2 + m^2$ |
| $VOA$ | $O$ | $a^2 = h^2 + R^2$ |
| $VMA$ | $M$ | $a^2 = g^2 + \left(\frac{\ell}{2}\right)^2$ |
| $OMA$ | $M$ | $R^2 = m^2 + \left(\frac{\ell}{2}\right)^2$ |

**Por que o ângulo em $M$ é reto no triângulo $VMA$?** Como a face lateral $VAB$ é um triângulo isósceles, a mediana $VM$ também é altura. Outra forma de ver: $VO$ é perpendicular à base e $OM \perp AB$; pelo **Teorema das três perpendiculares** (notebook 03), $VM \perp AB$.

**Apótema e raio das bases mais comuns** (módulo 04)

| Base | $m$ | $R$ |
|---|---|---|
| triângulo equilátero | $\dfrac{\ell\sqrt{3}}{6}$ | $\dfrac{\ell\sqrt{3}}{3}$ |
| quadrado | $\dfrac{\ell}{2}$ | $\dfrac{\ell\sqrt{2}}{2}$ |
| hexágono regular | $\dfrac{\ell\sqrt{3}}{2}$ | $\ell$ |

**Dois ângulos importantes**
- O **diedro** entre uma face lateral e a base (notebook 05): sua seção reta é o ângulo $V\hat{M}O$, com $\tan\theta = \dfrac{h}{m}$. É a **inclinação** da face, a mesma do telhado de uma casa com telhado em forma de pirâmide.
- O ângulo entre uma **aresta lateral** e a base (notebook 04): é o ângulo $V\hat{A}O$, com $\tan\varphi = \dfrac{h}{R}$.

> 🕵️ **Curiosidade histórica.** A **Grande Pirâmide de Gizé** (pirâmide de Quéops), no Egito, foi construída há cerca de 4.500 anos. É uma pirâmide quadrangular regular com base de aproximadamente 230 m de lado e altura original de aproximadamente 146 m. A inclinação das suas faces é de quase 52°. Por quase quatro mil anos, ela foi a construção mais alta feita pelo ser humano.

🕹️ Escolha um dos quatro triângulos para destacá-lo na pirâmide.

In [4]:
def explorar_relacoes(
    lados_da_base: int, lado: float, altura: float, triangulo: str
) -> None:
    """Destaca um dos quatro triângulos retângulos e calcula as medidas da pirâmide."""
    vertices, faces = u3d.piramide_regular(lados_da_base, lado, altura)
    V, A, B = vertices[-1], vertices[0], vertices[1]
    centro = (0, 0, 0)  # o ponto O, centro da base
    M = tuple(np.mean([A, B], axis=0))  # o ponto médio de AB

    m = lado / (2 * math.tan(math.pi / lados_da_base))      # apótema da base
    R = lado / (2 * math.sin(math.pi / lados_da_base))      # raio da base
    g = math.sqrt(altura**2 + m**2)                         # apótema da pirâmide (VOM)
    a = math.sqrt(altura**2 + R**2)                         # aresta lateral (VOA)
    diedro = math.degrees(math.atan(altura / m))            # face lateral com a base
    angulo_aresta = math.degrees(math.atan(altura / R))     # aresta lateral com a base

    # Para cada triângulo: o vértice do ângulo reto e os outros dois vértices.
    triangulos = {
        "VOM": (centro, V, M),
        "VOA": (centro, V, A),
        "VMA": (M, V, A),
        "OMA": (M, centro, A),
    }
    # Para cada triângulo: a relação, a hipotenusa e os dois catetos.
    relacoes = {
        "VOM": ("g² = h² + m²", g, altura, m),
        "VOA": ("a² = h² + R²", a, altura, R),
        "VMA": ("a² = g² + (ℓ/2)²", a, g, lado / 2),
        "OMA": ("R² = m² + (ℓ/2)²", R, m, lado / 2),
    }
    reto, ponto1, ponto2 = triangulos[triangulo]

    fig = u3d.criar_figura(titulo=f"Pirâmide regular: triângulo {triangulo}")
    fig = u3d.desenhar_poliedro(fig, vertices, faces, opacidade=0.2)
    fig = u3d.desenhar_poligono(
        fig, [reto, ponto1, ponto2], cor=u3d.CORES["destaque"], opacidade=0.5
    )
    # O quadradinho precisa caber no triângulo, mesmo com a base pequena.
    fig = u3d.desenhar_angulo_reto(
        fig, reto, ponto1, ponto2, cor=u3d.CORES["destaque"], tamanho=min(0.3, 0.4 * m)
    )
    for ponto, nome in ((V, "V"), (centro, "O"), (A, "A"), (B, "B"), (M, "M")):
        fig = u3d.desenhar_ponto(fig, ponto, rotulo=nome)
    u3d.mostrar(fig)

    print(f"Apótema da base:      m = {m:.2f}")
    print(f"Raio da base:         R = {R:.2f}")
    print(f"Apótema da pirâmide:  g = {g:.2f}")
    print(f"Aresta lateral:       a = {a:.2f}")
    print(f"Diedro face–base:     θ = {diedro:.2f}°")
    print(f"Ângulo aresta–base:   φ = {angulo_aresta:.2f}°")
    relacao, hipotenusa, cateto1, cateto2 = relacoes[triangulo]
    print(
        f"Triângulo {triangulo}: {relacao} → {hipotenusa:.2f}² = {cateto1:.2f}² + "
        f"{cateto2:.2f}² → {hipotenusa**2:.2f} = {cateto1**2 + cateto2**2:.2f}"
    )
    print(
        f"Conferência: a = {u3d.medir_distancia(V, A):.2f}; "
        f"g = {u3d.medir_distancia(V, M):.2f}; "
        f"θ = {u3d.medir_angulo_do_diedro(A, B, V, centro):.2f}°"
    )


interact(
    explorar_relacoes,
    lados_da_base=widgets.Dropdown(
        options=[3, 4, 6], value=4, description="lados da base"
    ),
    lado=widgets.FloatSlider(
        min=1, max=4, step=0.5, value=2, description="lado", continuous_update=False
    ),
    altura=widgets.FloatSlider(
        min=1, max=5, step=0.5, value=3, description="altura",
        continuous_update=False,
    ),
    triangulo=widgets.Dropdown(
        options=["VOM", "VOA", "VMA", "OMA"], value="VOM", description="triângulo"
    ),
);

interactive(children=(Dropdown(description='lados da base', index=1, options=(3, 4, 6), value=4), FloatSlider(…

**Pare e pense**

1. Numa pirâmide quadrangular regular, a aresta da base mede 6 e a altura mede 4. Quanto mede o apótema da pirâmide?
2. Na pirâmide de Quéops (base de 230 m, altura de 146 m), qual é a inclinação aproximada das faces?

<details>
<summary>Ver respostas</summary>

1. $m = 3$, então $g = \sqrt{4^2 + 3^2} = 5$.
2. $m = 115$ m, então $\tan\theta = \frac{146}{115} \approx 1{,}27$ e $\theta \approx 51{,}8°$.
</details>

## 3. Área lateral e área total

> 📌 **Definição.** A **área lateral** $S_\ell$ é a soma das áreas das faces laterais. A **área total** é $S_t = S_\ell + S_b$.

Numa pirâmide regular de base com $n$ lados, as $n$ faces laterais são triângulos congruentes de base $\ell$ e altura $g$:
$$S_\ell = n \cdot \frac{\ell \cdot g}{2} = p \cdot g$$
em que $p = \dfrac{n\ell}{2}$ é o **semiperímetro** da base.

> ⚠️ **Erro comum.** Usar a **altura** $h$ da pirâmide no lugar do **apótema** $g$ ao calcular a área das faces. A altura de cada triângulo lateral é $g$, que fica **sobre** a face; $h$ fica **dentro** da pirâmide.

**Aplicações:** a lona de uma barraca em forma de pirâmide, as telhas de um telhado piramidal, o vidro de uma cobertura. A **Pirâmide do Louvre**, em Paris, inaugurada em 1989, é uma pirâmide quadrangular regular de vidro e metal, com base de aproximadamente 35 m e altura de aproximadamente 21,6 m.

🕹️ Dobre a planificação e compare as áreas.

In [5]:
def planificar_piramide(
    lados_da_base: int, lado: float, altura: float, dobra: float
) -> None:
    """Dobra a planificação de uma pirâmide regular e calcula as suas áreas."""
    planificacao = u3d.planificacao_de_piramide_regular(lados_da_base, lado, altura)
    faces = u3d.dobrar_planificacao(planificacao, dobra)
    # A base numa cor e os triângulos laterais em outra.
    cores = [u3d.CORES["plano_secundario"]] + [u3d.CORES["plano"]] * lados_da_base
    fig = u3d.criar_figura(titulo="Planificação da pirâmide regular")
    fig = u3d.desenhar_faces(fig, faces, cores=cores)
    if dobra == 0:
        fig = u3d.definir_camera(fig, olho=(0, 0, 2.5))  # vista de cima
    u3d.mostrar(fig)

    m = lado / (2 * math.tan(math.pi / lados_da_base))
    g = math.sqrt(altura**2 + m**2)
    p = lados_da_base * lado / 2
    area_lateral = p * g
    area_base = p * m                  # área do polígono regular = semiperímetro × apótema
    area_total = area_lateral + area_base

    print(f"Apótema da base: m = {m:.2f}; apótema da pirâmide: g = {g:.2f}")
    print(f"Área lateral: S_ℓ = p · g = {p:g} × {g:.2f} = {area_lateral:.2f}")
    print(f"Área da base: S_b = p · m = {p:g} × {m:.2f} = {area_base:.2f}")
    print(f"Área total: S_t = S_ℓ + S_b = {area_total:.2f}")
    solido = u3d.piramide_regular(lados_da_base, lado, altura)
    print(f"Conferência: área total = {u3d.medir_area_total(*solido):.2f}")


interact(
    planificar_piramide,
    lados_da_base=widgets.IntSlider(
        min=3, max=8, step=1, value=4, description="lados da base",
        continuous_update=False,
    ),
    lado=widgets.FloatSlider(
        min=1, max=4, step=0.5, value=2, description="lado", continuous_update=False
    ),
    altura=widgets.FloatSlider(
        min=1, max=5, step=0.5, value=3, description="altura",
        continuous_update=False,
    ),
    dobra=widgets.FloatSlider(
        min=0, max=1, step=0.1, value=0, description="Dobrar", continuous_update=False
    ),
);

interactive(children=(IntSlider(value=4, continuous_update=False, description='lados da base', max=8, min=3), …

**Pare e pense**

1. Calcule a área lateral e a área total da pirâmide quadrangular regular de aresta da base 6 e altura 4.
2. Na planificação aberta, os triângulos laterais parecem "altos". A altura deles é $h$ ou $g$?

<details>
<summary>Ver respostas</summary>

1. $g = 5$; $S_\ell = 12 \cdot 5 = 60$; $S_b = 36$; $S_t = 96$.
2. É $g$, o apótema da pirâmide. Na planificação, a altura $h$ nem aparece: ela fica "escondida" dentro do sólido montado.
</details>

## 4. Volume da pirâmide

A demonstração tem três passos.

### Passo 1: seções paralelas à base

Corte uma pirâmide de altura $h$ por um plano paralelo à base, a uma distância $d$ do vértice. A seção é um polígono **semelhante** à base, com razão de semelhança $\frac{d}{h}$ (todas as arestas laterais são divididas na mesma razão, como no Teorema de Tales no espaço, notebook 02). Como áreas de figuras semelhantes estão na razão do **quadrado** da razão de semelhança:
$$S_{\text{seção}} = S_b \cdot \left(\frac{d}{h}\right)^2$$

> 📎 **Conexão.** Essa propriedade será estudada com mais detalhes no notebook 14 (Sólidos semelhantes e troncos).

### Passo 2: pirâmides de mesma base e mesma altura têm o mesmo volume

Considere duas pirâmides com bases de mesma área $S_b$ e mesma altura $h$, apoiadas num mesmo plano. Pelo passo 1, a uma distância $d$ do vértice, **as duas** seções têm área $S_b \left(\frac{d}{h}\right)^2$. Pelo **Princípio de Cavalieri**, as duas pirâmides têm o mesmo volume. Isso vale mesmo que uma seja oblíqua e a outra regular, e mesmo que as bases tenham formatos diferentes.

### Passo 3: o prisma triangular cortado em três

Um prisma triangular $ABC\text{-}DEF$ (com $D$, $E$, $F$ acima de $A$, $B$, $C$) pode ser cortado em **três pirâmides triangulares**, todas com vértice $F$:
- $P_1$ = pirâmide $F\text{-}ABC$;
- $P_2$ = pirâmide $F\text{-}ABE$;
- $P_3$ = pirâmide $F\text{-}ADE$.

As três têm o **mesmo volume**:
- $P_1$ e $P_3$: $P_3$ também pode ser vista como a pirâmide de vértice $A$ e base $DEF$. As bases $ABC$ e $DEF$ são congruentes e as alturas são iguais (a altura do prisma). Pelo passo 2, os volumes são iguais.
- $P_2$ e $P_3$: as bases $ABE$ e $ADE$ são as duas metades do paralelogramo $ABED$ (cortado pela diagonal $AE$): têm a mesma área e estão no mesmo plano. Com o mesmo vértice $F$, as alturas também são iguais. Pelo passo 2, os volumes são iguais.

Então cada pirâmide tem **um terço** do volume do prisma: $\frac{1}{3} S_b h$.

### Conclusão

Toda pirâmide pode ser dividida em pirâmides triangulares de mesma altura (basta dividir a base em triângulos). Somando:

**Teorema**
> O volume de uma pirâmide é **um terço** do produto da área da base pela altura:
> $$\boxed{\text{volume} = \frac{1}{3} S_b \cdot h}$$

🕹️ Separe as três pirâmides do prisma no widget abaixo.

In [6]:
# Os seis vértices do prisma triangular: D, E e F ficam 3 unidades acima de A, B e C.
A, B, C = (0, 0, 0), (2, 0, 0), (0, 2, 0)
D, E, F = (0, 0, 3), (2, 0, 3), (0, 2, 3)


def separar_prisma(afastamento: float, mostrar_prisma: bool) -> None:
    """Afasta as três pirâmides em que o prisma triangular se decompõe."""
    pecas = [
        u3d.piramide([A, B, C], F),
        u3d.piramide([A, B, E], F),
        u3d.piramide([A, D, E], F),
    ]
    afastadas = u3d.afastar_pecas(pecas, afastamento)
    cores = [
        u3d.CORES["plano"], u3d.CORES["plano_secundario"],
        u3d.CORES["plano_terciario"],
    ]
    nomes = [["A", "B", "C", "F"], ["A", "B", "E", "F"], ["A", "D", "E", "F"]]

    fig = u3d.criar_figura(titulo="O prisma triangular cortado em três pirâmides")
    for peca, cor, rotulos in zip(afastadas, cores, nomes, strict=True):
        fig = u3d.desenhar_poliedro(fig, *peca, cor=cor, opacidade=0.6, rotulos=rotulos)
    if mostrar_prisma:
        fig = u3d.desenhar_poliedro(fig, *u3d.prisma([A, B, C], (0, 0, 3)), opacidade=0.05)
    u3d.mostrar(fig)

    area_base = 2 * 2 / 2                 # triângulo ABC
    altura = 3
    volume_prisma = area_base * altura
    volume_cada_piramide = volume_prisma / 3

    print(f"Volume do prisma: S_b · h = {area_base:g} × {altura} = {volume_prisma:g}")
    print(f"Volume de cada pirâmide: {volume_prisma:g} / 3 = {volume_cada_piramide:g}")
    conferencia = ", ".join(f"{u3d.medir_volume(*peca):.4f}" for peca in pecas)
    print(f"Conferência das três pirâmides: {conferencia}")


interact(
    separar_prisma,
    afastamento=widgets.FloatSlider(
        min=0, max=2, step=0.25, value=1, description="afastamento",
        continuous_update=False,
    ),
    mostrar_prisma=widgets.Checkbox(
        value=False, description="Mostrar o prisma inteiro"
    ),
);

interactive(children=(FloatSlider(value=1.0, continuous_update=False, description='afastamento', max=2.0, step…

### Vendo o $\frac{1}{3}$ aparecer com fatias

Também dá para "ver" o um terço com fatias, como no notebook 09. Aproxime a pirâmide por uma escada de placas: a placa de baixo tem a área da base, e as de cima vão encolhendo. A soma dos volumes das placas fica um pouco **acima** do volume verdadeiro (as placas "sobram" para fora da pirâmide), mas a diferença diminui quando as placas ficam mais finas.

🕹️ Aumente o número de fatias e veja a soma se aproximar do volume exato.

In [ ]:
def somar_fatias(numero_de_fatias: int) -> float:
    """Soma os volumes das placas que aproximam a pirâmide de lado 2 e altura 3."""
    espessura = 3 / numero_de_fatias
    soma = 0
    for k in range(numero_de_fatias):
        lado_da_fatia = 2 * (1 - k / numero_de_fatias)
        soma = soma + lado_da_fatia**2 * espessura
    return soma


volume_exato = (1/3) * 2 * 2 * 3


def fatiar_piramide(numero_de_fatias: int) -> None:
    """Desenha a escada de placas e compara a soma com o volume exato."""
    base = u3d.poligono_regular(4, lado=2)
    fatias = u3d.fatias_de_piramide(base, (0, 0, 3), numero_de_fatias)
    espessura = 3 / numero_de_fatias
    soma = somar_fatias(numero_de_fatias)

    fig = u3d.criar_figura(titulo="A pirâmide como uma escada de placas")
    fig = u3d.desenhar_poliedro(fig, *u3d.piramide_regular(4, 2, 3), opacidade=0.15)
    fig = u3d.desenhar_pilha(fig, fatias, espessura)
    u3d.mostrar(fig)

    print(f"Espessura de cada placa: 3 / {numero_de_fatias} = {espessura:.4f}")
    print(f"Soma das placas: {soma:.4f}")
    print(f"Volume exato: (1/3) · 4 · 3 = {volume_exato:.4f}")
    print(f"Diferença: {soma - volume_exato:.4f}")


interact(
    fatiar_piramide,
    numero_de_fatias=widgets.IntSlider(
        min=2, max=60, step=2, value=6, description="fatias",
        continuous_update=False,
    ),
);

print(f"{'fatias':>8}{'soma':>12}{'diferença':>12}")
for numero in (1, 2, 5, 10, 50, 100, 1000):
    soma = somar_fatias(numero)
    print(f"{numero:>8}{soma:>12.4f}{soma - volume_exato:>12.4f}")

interactive(children=(IntSlider(value=6, continuous_update=False, description='fatias', max=60, min=2, step=2)…

  fatias        soma   diferença
       1     12.0000      8.0000
       2      7.5000      3.5000
       5      5.2800      1.2800
      10      4.6200      0.6200
      50      4.1208      0.1208
     100      4.0602      0.0602
    1000      4.0060      0.0060


**Pare e pense**

1. Qual é o volume da pirâmide quadrangular regular de aresta da base 6 e altura 4?
2. Uma pirâmide e um prisma têm a mesma base e o mesmo volume. Qual é a relação entre as alturas?

<details>
<summary>Ver respostas</summary>

1. $\frac{1}{3} \cdot 36 \cdot 4 = 48$.
2. A altura da pirâmide é o triplo da altura do prisma: $\frac{1}{3} S_b h_{\text{pir}} = S_b h_{\text{pri}}$.
</details>

## 5. O tetraedro regular

O **tetraedro regular** é a pirâmide triangular com as **seis arestas iguais** (de medida $a$). Ele é um dos cinco sólidos de Platão (notebook 07): qualquer uma das suas faces pode ser a base.

**Altura.** A base é um triângulo equilátero de lado $a$, com raio $R = \frac{a\sqrt{3}}{3}$. No triângulo $VOA$, retângulo em $O$:
$$h^2 = a^2 - R^2 = a^2 - \frac{a^2}{3} = \frac{2a^2}{3} \quad\Longrightarrow\quad \boxed{h = \frac{a\sqrt{6}}{3}}$$

**Área total.** São 4 triângulos equiláteros de lado $a$:
$$\boxed{S_t = 4 \cdot \frac{a^2\sqrt{3}}{4} = a^2\sqrt{3}}$$

**Volume.**
$$\text{volume} = \frac{1}{3} \cdot \frac{a^2\sqrt{3}}{4} \cdot \frac{a\sqrt{6}}{3} = \frac{a^3\sqrt{18}}{36} \quad\Longrightarrow\quad \boxed{\text{volume} = \frac{a^3\sqrt{2}}{12}}$$

> 💡 **Dica.** A primeira embalagem da **Tetra Pak**, lançada na Suécia nos anos 1950, tinha a forma de um tetraedro (daí o nome "Tetra"). O formato era fácil de fabricar a partir de um tubo de papel cartonado, fechado com duas soldas perpendiculares.

🕹️ Mexa na aresta e confira as fórmulas.

In [ ]:
def explorar_tetraedro(aresta: float, mostrar_triangulo: bool) -> None:
    """Desenha o tetraedro regular e confere a altura, a área e o volume."""
    altura = aresta * math.sqrt(6) / 3
    area_total = aresta**2 * math.sqrt(3)
    volume = aresta**3 * math.sqrt(2) / 12

    vertices, faces = u3d.piramide_regular(3, aresta, altura)
    V, A = vertices[-1], vertices[0]
    centro = (0, 0, 0)  # o ponto O, centro da base

    fig = u3d.criar_figura(titulo="O tetraedro regular")
    fig = u3d.desenhar_poliedro(fig, vertices, faces, opacidade=0.3)
    if mostrar_triangulo:
        fig = u3d.desenhar_poligono(
            fig, [V, centro, A], cor=u3d.CORES["destaque"], opacidade=0.5
        )
        fig = u3d.desenhar_angulo_reto(
            fig, centro, V, A, cor=u3d.CORES["destaque"], tamanho=0.2 * aresta
        )
        for ponto, nome in ((V, "V"), (centro, "O"), (A, "A")):
            fig = u3d.desenhar_ponto(fig, ponto, rotulo=nome)
    u3d.mostrar(fig)

    print(f"Altura: h = a√6 / 3 = {altura:.4f}")
    print(f"Área total: S_t = a²√3 = {area_total:.4f}")
    print(f"Volume: a³√2 / 12 = {volume:.4f}")

    # No tetraedro, quaisquer dois vértices são ligados por uma aresta.
    comprimentos = [
        u3d.medir_distancia(vertices[i], vertices[j])
        for i in range(4)
        for j in range(i + 1, 4)
    ]
    print("Conferência das 6 arestas:", ", ".join(f"{c:.4f}" for c in comprimentos))
    print(f"Conferência da área total: {u3d.medir_area_total(vertices, faces):.4f}")
    print(f"Conferência do volume: {u3d.medir_volume(vertices, faces):.4f}")


interact(
    explorar_tetraedro,
    aresta=widgets.FloatSlider(
        min=1, max=6, step=0.5, value=2, description="aresta",
        continuous_update=False,
    ),
    mostrar_triangulo=widgets.Checkbox(
        value=False, description="Mostrar triângulo VOA"
    ),
);

interactive(children=(FloatSlider(value=2.0, continuous_update=False, description='aresta', max=6.0, min=1.0, …

**Pare e pense**

1. Qual é a altura de um tetraedro regular de aresta 6?
2. Se a aresta de um tetraedro regular dobra, o que acontece com o volume?

<details>
<summary>Ver respostas</summary>

1. $h = \frac{6\sqrt{6}}{3} = 2\sqrt{6} \approx 4{,}90$.
2. Fica multiplicado por $2^3 = 8$ (o volume depende de $a^3$).
</details>

## Resumo

**Pirâmide:** base poligonal e vértice fora do plano da base. Base com $n$ lados: $n + 1$ vértices, $2n$ arestas, $n + 1$ faces.

**Pirâmide regular:** base regular e vértice projetado no centro da base. Faces laterais: triângulos isósceles congruentes.

**Relações métricas:** $g^2 = h^2 + m^2$; $a^2 = h^2 + R^2$; $a^2 = g^2 + \left(\frac{\ell}{2}\right)^2$; $R^2 = m^2 + \left(\frac{\ell}{2}\right)^2$.
Diedro face–base: $\tan\theta = \frac{h}{m}$. Aresta lateral–base: $\tan\varphi = \frac{h}{R}$.

**Áreas:** $S_\ell = p \cdot g$ (semiperímetro da base × apótema da pirâmide); $S_t = S_\ell + S_b$.

**Seção paralela à base** a distância $d$ do vértice: $S = S_b \left(\frac{d}{h}\right)^2$.

**Volume:** $\dfrac{1}{3} S_b \cdot h$. Pirâmides de mesma base (em área) e mesma altura têm o mesmo volume. O prisma triangular se divide em 3 pirâmides de mesmo volume.

**Tetraedro regular** de aresta $a$: $h = \dfrac{a\sqrt{6}}{3}$; $S_t = a^2\sqrt{3}$; volume $= \dfrac{a^3\sqrt{2}}{12}$.

## Exercícios

Responda preenchendo as variáveis indicadas em cada célula (troque cada `...` pela sua resposta) e rode a célula de verificação logo abaixo. Os exercícios estão em três níveis: **básico**, **intermediário** e **desafio**.

O gabarito comentado está em [`0610b_piramide_gabarito.ipynb`](0610b_piramide_gabarito.ipynb). Tente sozinho antes de olhar!

### Básico

In [ ]:
# Exercício Básico 1: verdadeiro ou falso? Troque cada ... por True ou False.
# (a) Toda pirâmide de base regular é uma pirâmide regular.
# (b) As faces laterais de uma pirâmide regular são triângulos isósceles congruentes.
# (c) Uma pirâmide hexagonal tem 12 arestas.
# (d) O volume de uma pirâmide é um terço do volume do prisma de mesma base e mesma
#     altura.
# (e) Uma pirâmide oblíqua tem volume menor que uma pirâmide regular de mesma base e
#     mesma altura.
respostas_ex1 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
    "e": ...,
}

print(respostas_ex1)

In [ ]:
# Verificação
gabarito_ex1 = {"a": False, "b": True, "c": True, "d": True, "e": False}
dicas_ex1 = {
    "a": "o vértice também precisa se projetar no centro da base (seção 1 - Pirâmide)",
    "b": "a definição de pirâmide regular (seção 1 - Pirâmide)",
    "c": "base com n lados: 2n arestas (seção 1 - Pirâmide)",
    "d": "a fórmula do volume (seção 4 - Volume da pirâmide)",
    "e": "o passo 2 da demonstração: mesma base e mesma altura dão o mesmo volume "
    "(seção 4 - Volume da pirâmide)",
}
for item, certa in gabarito_ex1.items():
    assert respostas_ex1[item] == certa, (
        f"Tente de novo, revise: o item ({item}) não está certo; {dicas_ex1[item]}."
    )
print("Certo!")

In [ ]:
# Exercício Básico 2: uma pirâmide quadrangular regular tem aresta da base 6 e altura 4.
# Calcule, em Python (use math.sqrt quando precisar):
# (a) o apótema da base;
# (b) o apótema da pirâmide;
# (c) a aresta lateral;
# (d) a área total;
# (e) o volume.
apotema_da_base_ex2 = ...
apotema_da_piramide_ex2 = ...
aresta_lateral_ex2 = ...
area_total_ex2 = ...
volume_ex2 = ...

print(
    apotema_da_base_ex2, apotema_da_piramide_ex2, aresta_lateral_ex2, area_total_ex2,
    volume_ex2,
)

In [ ]:
# Verificação
respostas_ex2 = [
    ("a", apotema_da_base_ex2, 3, "na base quadrada, m = ℓ / 2",
     "seção 2 - Relações métricas da pirâmide regular"),
    ("b", apotema_da_piramide_ex2, 5, "use o triângulo VOM: g² = h² + m²",
     "seção 2 - Relações métricas da pirâmide regular"),
    ("c", aresta_lateral_ex2, math.sqrt(34), "use o triângulo VOA ou o VMA",
     "seção 2 - Relações métricas da pirâmide regular"),
    ("d", area_total_ex2, 96, "S_t = p · g + S_b, com g no lugar de h",
     "seção 3 - Área lateral e área total"),
    ("e", volume_ex2, 48, "o volume é (1/3) · S_b · h",
     "seção 4 - Volume da pirâmide"),
]
for item, resposta, certa, dica, secao in respostas_ex2:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), {dica} ({secao})."
    )
print("Certo!")

In [ ]:
# Exercício Básico 3: contando elementos. Uma pirâmide tem base pentagonal. Quantos
# ela tem de:
# (a) vértices;
# (b) arestas;
# (c) faces?
vertices_ex3 = ...
arestas_ex3 = ...
faces_ex3 = ...

print(vertices_ex3, arestas_ex3, faces_ex3)

In [ ]:
# Verificação
respostas_ex3 = [("a", vertices_ex3, 6), ("b", arestas_ex3, 10), ("c", faces_ex3, 6)]
for item, resposta, certa in respostas_ex3:
    assert resposta == certa, (
        f"Tente de novo, revise: no item ({item}), base com n lados tem n + 1 vértices, "
        "2n arestas e n + 1 faces (seção 1 - Pirâmide)."
    )
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 4: a pirâmide de Quéops. Considere a Grande Pirâmide como uma
# pirâmide quadrangular regular com base de 230 m de lado e altura de 146 m. Calcule,
# em Python:
# (a) o volume, em m³;
# (b) a inclinação das faces, em graus (use math.atan e math.degrees).
volume_ex4 = ...
inclinacao_ex4 = ...

print(volume_ex4, inclinacao_ex4)

In [ ]:
# Verificação
assert volume_ex4 is not ... and abs(volume_ex4 - 230**2 * 146 / 3) <= 1000, (
    "Tente de novo, revise: no item (a), o volume é (1/3) · S_b · h, com "
    "S_b = 230² (seção 4 - Volume da pirâmide)."
)
assert (
    inclinacao_ex4 is not ...
    and abs(inclinacao_ex4 - math.degrees(math.atan(146 / 115))) <= 0.1
), (
    "Tente de novo, revise: no item (b), use o triângulo VOM: tan θ = h / m, com "
    "m = 115 (seção 2 - Relações métricas da pirâmide regular)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 5: o tetraedro regular. Para um tetraedro regular de aresta 6,
# calcule:
# (a) a altura;
# (b) a área total;
# (c) o volume.
altura_ex5 = ...
area_total_ex5 = ...
volume_ex5 = ...

print(altura_ex5, area_total_ex5, volume_ex5)

In [ ]:
# Verificação
respostas_ex5 = [
    ("a", altura_ex5, 2 * math.sqrt(6)),
    ("b", area_total_ex5, 36 * math.sqrt(3)),
    ("c", volume_ex5, 18 * math.sqrt(2)),
]
for item, resposta, certa in respostas_ex5:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), h = a√6/3, S = a²√3 e "
        "volume = a³√2/12 (seção 5 - O tetraedro regular)."
    )
print("Certo!")

In [ ]:
# Exercício Intermediário 6: o vidro do Louvre. Considere a Pirâmide do Louvre como uma
# pirâmide quadrangular regular com base de 35 m e altura de 21,6 m. Calcule:
# (a) o apótema da pirâmide;
# (b) a área lateral (a área coberta pelo vidro, sem contar a base).
apotema_ex6 = ...
area_lateral_ex6 = ...

print(apotema_ex6, area_lateral_ex6)

In [ ]:
# Verificação
assert (
    apotema_ex6 is not ...
    and abs(apotema_ex6 - math.sqrt(21.6**2 + 17.5**2)) <= 0.5
), (
    "Tente de novo, revise: no item (a), g² = h² + m², com m = 17,5 "
    "(seção 2 - Relações métricas da pirâmide regular)."
)
assert (
    area_lateral_ex6 is not ...
    and abs(area_lateral_ex6 - 70 * math.sqrt(21.6**2 + 17.5**2)) <= 0.5
), (
    "Tente de novo, revise: no item (b), S = p · g, com p = 70 "
    "(seção 3 - Área lateral e área total)."
)
print("Certo!")

### Desafio

In [ ]:
# Exercício Desafio 7: três pirâmides dentro do cubo. No cubo ABCDEFGH de aresta 2,
# considere as pirâmides de vértice G e bases ABCD, ABFE e ADHE (as três faces que NÃO
# contêm G).
# (a) Qual é o volume de cada uma?
# (b) As três são congruentes? (True ou False)
# (c) Elas preenchem o cubo inteiro? (True ou False)
volume_ex7 = ...
congruentes_ex7 = ...
preenchem_ex7 = ...

print(volume_ex7, congruentes_ex7, preenchem_ex7)

In [ ]:
# Verificação
assert volume_ex7 is not ... and abs(volume_ex7 - 8 / 3) <= 0.01, (
    "Tente de novo, revise: no item (a), cada pirâmide tem base de área 4 e altura 2, "
    "a distância de G à face oposta (seção 4 - Volume da pirâmide)."
)
assert congruentes_ex7 is True, (
    "Tente de novo, revise: no item (b), compare as medidas: cada pirâmide tem uma base "
    "quadrada de lado 2 e uma aresta lateral de 2 perpendicular à base "
    "(seção 1 - Pirâmide)."
)
assert preenchem_ex7 is True, (
    "Tente de novo, revise: no item (c), some os três volumes e compare com o volume "
    "do cubo (seção 4 - Volume da pirâmide)."
)
print("Certo!")

# As três pirâmides afastadas e, depois, juntas.
cubo = u3d.vertices_do_cubo()
pecas = [
    u3d.piramide([cubo[letra] for letra in base], cubo["G"])
    for base in ("ABCD", "ABFE", "ADHE")
]
cores = [
    u3d.CORES["plano"], u3d.CORES["plano_secundario"], u3d.CORES["plano_terciario"],
]
for titulo, distancia in (("Afastadas", 1.5), ("Juntas: um cubo", 0)):
    fig = u3d.criar_figura(titulo=titulo)
    for peca, cor in zip(u3d.afastar_pecas(pecas, distancia), cores, strict=True):
        fig = u3d.desenhar_poliedro(fig, *peca, cor=cor, opacidade=0.6)
    u3d.mostrar(fig)
print('Esta é outra forma de "ver" o 1/3: três pirâmides iguais formam um cubo.')

In [ ]:
# Exercício Desafio 8: quantas fatias? Para uma pirâmide de base com área 1 e altura 1
# (volume exato 1/3), a aproximação por N placas da seção 4 é a função abaixo.
# Encontre o MENOR N para o qual a diferença entre soma_das_placas(N) e 1/3 fica menor
# que 0,001.
def soma_das_placas(N: int) -> float:
    """Soma os volumes das N placas que aproximam a pirâmide."""
    soma = 0
    for k in range(N):
        soma = soma + (1 - k / N) ** 2 * (1 / N)
    return soma


# Complete o laço: tire o # das linhas abaixo e troque o ... pela condição de continuar
# procurando. No fim, guarde a resposta em menor_N_ex8.
# N = 1
# while ...:  # complete aqui
#     N = N + 1
menor_N_ex8 = ...

print(menor_N_ex8)

In [ ]:
# Verificação
assert menor_N_ex8 == 501, (
    "Tente de novo, revise: use um laço while que aumenta N de 1 em 1 até a diferença "
    "ficar menor que 0,001 (seção 4 - Volume da pirâmide)."
)
print("Certo!")
print(
    "Somar fatias cada vez mais finas para obter um valor exato é a ideia que você vai "
    "formalizar no módulo 11 (Cálculo)."
)

## Conexões

**Isso depende de:**
- [`0603a_perpendicularidade.ipynb`](0603a_perpendicularidade.ipynb) (três perpendiculares: o apótema da pirâmide)
- [`0605a_diedros.ipynb`](0605a_diedros.ipynb) (diedro entre a face lateral e a base) e [`0606a_triedros_e_angulos_poliedricos.ipynb`](0606a_triedros_e_angulos_poliedricos.ipynb) (a pirâmide ilimitada é um ângulo poliédrico)
- [`0607a_poliedros_convexos.ipynb`](0607a_poliedros_convexos.ipynb) (Relação de Euler e tetraedro)
- [`0609a_volume_e_principio_de_cavalieri.ipynb`](0609a_volume_e_principio_de_cavalieri.ipynb) (Princípio de Cavalieri e volume do prisma)
- [`0416a_triangulos_retangulos.ipynb`](../04_geometria_plana/0416a_triangulos_retangulos.ipynb) (Pitágoras), [`0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb`](../04_geometria_plana/0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb) (polígonos regulares e apótema) e [`0415a_semelhanca_de_triangulos_e_potencia_de_ponto.ipynb`](../04_geometria_plana/0415a_semelhanca_de_triangulos_e_potencia_de_ponto.ipynb) (semelhança)

**Isso será usado em:**
- 📌 Cone (o cone como "pirâmide de base circular", com o mesmo $\frac{1}{3}$)
- 📌 Esfera (a dedução da área da esfera usa pirâmides)
- 📌 Sólidos semelhantes e troncos (seção paralela à base e tronco de pirâmide)
- 📌 Inscrição e circunscrição: esfera e poliedros (esfera e tetraedro regular)
- 📌 Limite, Integral e Derivada (volumes por soma de fatias)